In [21]:
import pandas as pd

In [22]:
data_file_url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

In [23]:
df = pd.read_csv(data_file_url)

### EDA

In [24]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [25]:
df.head().T

,0,1,2,3,4
lead_source,organic_search,social_media,referral,paid_ads,referral
industry,technology,technology,retail,manufacturing,manufacturing
employment_status,employed,employed,employed,student,student
location,europe,south_america,europe,NaN,north_america
annual_income,88160.0,72688.0,44697.0,NaN,24062.0
number_of_courses_viewed,3,3,3,3,4
interaction_count,4,7,4,6,5
lead_score,0.64,0.72,0.58,0.57,0.62
converted,1,1,1,0,1


In [41]:
categorical_columns = list(df.select_dtypes(include=['str']).columns)
categorical_columns

['lead_source', 'industry', 'employment_status', 'location']

In [42]:
numerical_columns = list(df.select_dtypes(include=['int64', 'float64']).columns)

# Delete the 'converted' value in numerical_columns
numerical_columns = [col for col in numerical_columns if col != 'converted']
numerical_columns

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [43]:
for col in categorical_columns:
    print(f"{col}: {df[col].nunique()} unique values")
    print(list(df[col].unique()))
    print()
    print()

lead_source: 6 unique values
['organic_search', 'social_media', 'referral', 'paid_ads', 'NA', 'events']


industry: 7 unique values
['technology', 'retail', 'manufacturing', 'education', 'healthcare', 'NA', 'finance']


employment_status: 5 unique values
['employed', 'student', 'NA', 'unemployed', 'self_employed']


location: 6 unique values
['europe', 'south_america', 'NA', 'north_america', 'africa', 'asia']




In [44]:
df[categorical_columns].isnull().sum()

lead_source          0
industry             0
employment_status    0
location             0
dtype: int64

In [45]:
# Replace categotical null values with "NA"
for col in categorical_columns:
    df[col] = df[col].fillna("NA")

In [46]:
df[numerical_columns].isnull().sum()

annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
dtype: int64

In [47]:
# Replace numerical null values with 0.0
for col in numerical_columns:
    df[col] = df[col].fillna(0.0)


### Q1

In [48]:
df.industry.mode()

0    technology
Name: industry, dtype: str

In [49]:
df["industry"].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

### Q2

In [50]:
# Correlation matrix for numerical columns
correlation_matrix = df[numerical_columns].corr()
correlation_matrix

# interaction_count and lead_score have a strong positive correlation of 0.9157

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [51]:
# Split the data
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

# Delete the target value "converted" from the training, validation and test datasets
y_train = df_train["converted"].values
y_val = df_val["converted"].values
y_test = df_test["converted"].values

del df_train["converted"]
del df_val["converted"]
del df_test["converted"]

### Q3

In [52]:
from sklearn.metrics import mutual_info_score

# Applying mutual information to all categorical features
# Round the mutual information score to 2 decimal places
def calculate_mutual_info(series):
    return mutual_info_score(df_full_train['converted'], series)

mi = df_full_train[categorical_columns].apply(calculate_mutual_info)
mi = mi.round(2)
mi.sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

### Q4

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer

# Use one-hot encoding for categorical features
dv = DictVectorizer(sparse=False)
train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

# Create a logistic regression model with the following parameters:
# solver='liblinear', C=1.0, max_iter=1000, random_state=42
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Evaluate the model on the validation set
val_dict = df_val[categorical_columns + numerical_columns].to_dict(orient='records')
X_val = dv.transform(val_dict)

y_val_pred = model.predict(X_val)

# Get Accuracy
y_val_accuracy = (y_val_pred == y_val).mean()
print(f"Validation Accuracy: {y_val_accuracy:.2f}")


Validation Accuracy: 0.65


In [57]:
# Eliminate each feature one by one and evaluate the model on the validation set
feature_importance = {}
features = categorical_columns + numerical_columns
for feature in features:
    # Create a new training and validation set without the feature
    new_features = [f for f in features if f != feature]
    df_train_new = df_train.drop(columns=[feature])
    df_val_new = df_val.drop(columns=[feature])
    
    # Use one-hot encoding for categorical features
    dv_new = DictVectorizer(sparse=False)
    train_dict_new = df_train_new[new_features].to_dict(orient='records')
    X_train_new = dv_new.fit_transform(train_dict_new)
    
    # Create a logistic regression model with the same parameters
    model_new = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_new.fit(X_train_new, y_train)
    
    # Evaluate the model on the validation set
    val_dict_new = df_val_new[new_features].to_dict(orient='records')
    X_val_new = dv_new.transform(val_dict_new)
    
    y_val_pred_new = model_new.predict(X_val_new)
    
    # Get Accuracy
    y_val_accuracy_new = (y_val_pred_new == y_val).mean()
    
    feature_importance[feature] = y_val_accuracy - y_val_accuracy_new
    print(f"Feature Dropped: {feature}, Accuracy Drop: {y_val_accuracy - y_val_accuracy_new}")


Feature Dropped: lead_source, Accuracy Drop: 0.0030000000000000027
Feature Dropped: industry, Accuracy Drop: 0.0
Feature Dropped: employment_status, Accuracy Drop: 0.0
Feature Dropped: location, Accuracy Drop: 0.0
Feature Dropped: annual_income, Accuracy Drop: -0.07899999999999996
Feature Dropped: number_of_courses_viewed, Accuracy Drop: 0.0020000000000000018
Feature Dropped: interaction_count, Accuracy Drop: 0.04400000000000004
Feature Dropped: lead_score, Accuracy Drop: 0.0010000000000000009


### Q6

In [59]:
# Training the model with C: [0.000001, 0.00001, 0.0001, 0.001]
C_values = [0.000001, 0.00001, 0.0001, 0.001]

# Use one-hot encoding for categorical features
dv = DictVectorizer(sparse=False)
train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

# Evaluate the model on the validation set
val_dict = df_val[categorical_columns + numerical_columns].to_dict(orient='records')
X_val = dv.transform(val_dict)

for C in C_values:
    # Create a logistic regression model with the following parameters:
    # solver='liblinear', C=C, max_iter=1000, random_state=42
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    y_val_pred = model.predict(X_val)
    
    # Get Accuracy
    y_val_accuracy = (y_val_pred == y_val).mean()
    print(f"C: {C}, Validation Accuracy: {y_val_accuracy:.3f}")

C: 1e-06, Validation Accuracy: 0.598
C: 1e-05, Validation Accuracy: 0.598
C: 0.0001, Validation Accuracy: 0.613
C: 0.001, Validation Accuracy: 0.645
